# Wind-tunnel stage-ratio audit
Read-only reproduction and formula checks. Cells were executed in order with Python; a Jupyter kernel was not available.
All source values remain observed. High starts at the first post-takeoff decrease (99% in the with-wind cohort); later integer-step plateaus remain included. Frozen baseline slopes are not refitted.


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'battery_normalization.py').exists())
BASE = ROOT/'analysis_results/wind_tunnel_stage_ratio_check_20260909'
OUT = BASE/'first_drop_recalculation'
sys.path.insert(0,str(ROOT/'output_py'))
from check_stage_ratio_sensitivity import calculate, stats
from audit_wind_tunnel_stage_ratios import sha
_, _, preland = calculate()
pairs=pd.read_csv(OUT/'paired_ratios.csv')
stages=pd.read_csv(OUT/'stage_rates.csv')
wind = pairs[pairs.wind_level.gt(0)]
print('Wind flights:', wind.source.nunique(), 'complete drone curves:', len(wind))


Wind flights: 16 complete drone curves: 43


In [2]:
for source, rows in stages.groupby('source'):
    raw=pd.read_csv(ROOT/source,usecols=['drone_name','elapsed_time','battery'])
    for _,s in rows.iterrows():
        g=raw[raw.drone_name.eq(s.drone_id)]
        for t,soc in [(s.start_s,s.soc_upper),(s.end_s,s.soc_lower)]:
            endpoint=g[np.isclose(g.elapsed_time,t,rtol=0,atol=1e-6)]
            assert len(endpoint)==1 and endpoint.battery.iloc[0]==soc
        expected=(s.soc_upper-s.soc_lower)*60/(s.end_s-s.start_s)
        assert np.isclose(expected,s.raw_rate_pp_min,rtol=1e-8)
        assert np.isclose(expected/s.baseline_rate_pp_min,s.k,rtol=1e-8)
manifest=json.loads((OUT/'manifest.json').read_text())
for source, expected in manifest['input_files_sha256'].items():
    assert sha(ROOT/source)==expected
print('All 282 raw SOC/time endpoints, 141 rates and source hashes passed.')


All 282 raw SOC/time endpoints, 141 rates and source hashes passed.


In [3]:
for stage in ['high','low']:
    recomputed=100*(wind[stage+'_k']/wind.medium_k-1)
    assert np.allclose(recomputed,wind[stage+'_difference_pct'])
    print(stage,stats(recomputed))
print('Low before first swarm landing:',stats(preland.loc[preland.included & preland.wind_level.gt(0),'difference_pct']))
print(wind[['experiment_id','drone_id','battery_id','high_difference_pct','low_difference_pct']].head(10).to_string(index=False))


high {'n': 43, 'median_signed_pct': 6.743154966855935, 'median_absolute_pct': 23.408236034103137, 'mean_absolute_pct': 26.55568677131435, 'min_pct': -71.36387981790315, 'max_pct': 71.99975086289628, 'within10': 10, 'within20': 20}
low {'n': 43, 'median_signed_pct': -2.8981536507216688, 'median_absolute_pct': 7.458860672086132, 'mean_absolute_pct': 6.521439278762985, 'min_pct': -16.898826665771736, 'max_pct': 9.36604252690647, 'within10': 35, 'within20': 43}
Low before first swarm landing: {'n': 37, 'median_signed_pct': -12.183333187569357, 'median_absolute_pct': 12.183333187569357, 'mean_absolute_pct': 12.108377475968432, 'min_pct': -22.24229768393059, 'max_pct': 9.366042527238472, 'within10': 16, 'within20': 31}
                      experiment_id drone_id battery_id  high_difference_pct  low_difference_pct
 wind_tunnel_column_75_head_lv1_001  drone_1        B11            10.570474           -2.898154
 wind_tunnel_column_75_head_lv1_001  drone_2        B10            28.128531       